In [1]:
import pandas as pd
import numpy as np
from func import clean_df

# model imports
from keras.models import Sequential
from keras.layers import Dense, Dropout
import keras.layers as layers

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

I0000 00:00:1774388820.270114    8003 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774388820.316722    8003 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774388822.356129    8003 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
data = pd.read_csv('./dataset/data.csv') # read the dataset
data = clean_df(df=data, drop_columns=['text']) # clean the dataset

print(len(data))
print(data['subject'].unique())
data.sample(10)

date
https://fedup.wpengine.com/wp-content/uploads/2015/04/hillarystreetart.jpg                                                                               2
https://fedup.wpengine.com/wp-content/uploads/2015/04/entitled.jpg                                                                                       2
https://100percentfedup.com/served-roy-moore-vietnamletter-veteran-sets-record-straight-honorable-decent-respectable-patriotic-commander-soldier/        1
https://100percentfedup.com/video-hillary-asked-about-trump-i-just-want-to-eat-some-pie/                                                                 1
https://100percentfedup.com/12-yr-old-black-conservative-whose-video-to-obama-went-viral-do-you-really-love-america-receives-death-threats-from-left/    1
MSNBC HOST Rudely Assumes Steel Worker Would Never Let His Son Follow in His Footsteps…He Couldn’t Be More Wrong [Video]                                 1
Name: count, dtype: int64
Rows dropped: 29938/39942 (74.95%)
1000

,label,title,subject,date,reporter,content,year,month,day
52,0,Americans Once Elected A President After He W...,News,2017-11-29,Unknown,After an awful campaign filled with hateful rh...,2017.0,11.0,29.0
920,0,Trump Condemned By Jewish Leaders In Poland A...,News,2017-07-06,Unknown,Republicans were crossing their fingers that D...,2017.0,7.0,6.0
4747,0,President Obama Warns Putin: Don’t Hack Our E...,News,2016-09-06,Unknown,"Recently, intelligence agencies, including the...",2016.0,9.0,6.0
6656,0,Paul Ryan Wants To Make This Healthcare Night...,News,2016-04-28,Unknown,"On Wednesday, Republican House Speaker Paul Ry...",2016.0,4.0,28.0
1486,0,"Ex-FBI Tears Trump To Pieces, Wonders ‘Who’s ...",News,2017-05-12,Unknown,"All Americans, and indeed the entire world, sh...",2017.0,5.0,12.0
6607,0,WATCH: Ted Cruz Gets His A** Handed To Him By...,News,2016-05-01,Unknown,Chuck Todd took Ted Cruz to the woodshed on Su...,2016.0,5.0,1.0
2078,0,Conservatives Blast Tomi Lahren For Being A H...,News,2017-03-20,Unknown,"Tomi Lahren, the overly-emotional, angry and v...",2017.0,3.0,20.0
7831,0,Bill Maher Acts Out Trump SOTU Address And PE...,News,2016-02-27,Unknown,Imagining what Donald Trump would be like as t...,2016.0,2.0,27.0
4133,0,Paul LePage Claims Election Will Be Rigged In...,News,2016-10-19,Unknown,It sounds like Paul LePage just claimed that h...,2016.0,10.0,19.0
7841,0,Scathing New Amnesty International Report Cal...,News,2016-02-26,Unknown,"On February 25, 1994, Israeli-Zionist terroris...",2016.0,2.0,26.0


In [3]:
mask = pd.to_datetime(data['date'], format='mixed', errors='coerce').isna()
print(data[mask]['date'].value_counts().head(20))

Series([], Name: count, dtype: int64)


In [4]:
data.sample(20)

,label,title,subject,date,reporter,content,year,month,day
3471,0,Teen Vogue Just Just SCHOOLED Major News Netw...,News,2016-12-10,Unknown,"Throughout the course of the election, the mai...",2016.0,12.0,10.0
2945,0,Rick Perry’s Confirmation Hearing Just Got Ex...,News,2017-01-19,Unknown,Things got a bit uncomfortable during Rick Per...,2017.0,1.0,19.0
4489,0,Watch Trump’s Campaign Manager Crash And Burn...,News,2016-09-26,Unknown,"Defending Donald Trump is no easy task, but ev...",2016.0,9.0,26.0
1029,0,Stephen Colbert Trolls Trump By Announcing Ru...,News,2017-06-24,Unknown,Is he really going to do it this time? With St...,2017.0,6.0,24.0
1409,0,Trump’s FCC Will Decimate Internet Freedom (V...,News,2017-05-19,Unknown,Republicans on the Federal Communications Com...,2017.0,5.0,19.0
6753,0,Curt Schilling Doubles Down On Transgender Ha...,News,2016-04-22,Unknown,After ESPN fired washed-up MLB pitcher Curt Sc...,2016.0,4.0,22.0
9922,0,ABC Shuts Down Conservative Tim Allen’s “Last ...,left-news,2017-05-13,Unknown,Tim Allen s hit sitcom Last Man Standing has...,2017.0,5.0,13.0
6968,0,"Meet Zari, The First Feminist Muppet From Afg...",News,2016-04-11,Unknown,This is awesome on so many levels it s hard to...,2016.0,4.0,11.0
7039,0,The DEA Will Deliver Some Big News About Mari...,News,2016-04-07,Unknown,The Drug Enforcement Agency released a 25-page...,2016.0,4.0,7.0
1736,0,New Research Shows Trump’s Border Wall Will B...,News,2017-04-19,Unknown,Build the wall! Build the wall! Supposedly n...,2017.0,4.0,19.0


In [5]:
# split the dataset into training and testing sets
x_data_lst = ['title', 'content']